Загружаю датасет с kaggle

In [7]:
import numpy as np
import pandas as pd

import kagglehub
import pandas as pd

# 1. Скачиваем датасет через kagglehub
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")

print("Путь к загруженным файлам:", path)





Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Путь к загруженным файлам: /kaggle/input/steam-games-dataset


Теперь считываю датасет

In [26]:
dataset = pd.read_csv(f"{path}/games.csv")
# Специальная обработка, так как в датасете эти 2 признака склеены в один: DiscountDLC count
dataset.rename(columns={'DiscountDLC count': 'DLC count'}, inplace=True)


Получаю размер датасета и количество признаков


In [21]:
dataset.shape

(125855, 39)

Количество классов - пусть будет 10 (то есть оценка от 1 до 10, насколько игра "крутая")

Пропущенные значения (в процентах):

In [10]:
print(dataset.isnull().sum().sum() / dataset.size * 100)

17.31426784384553


Несбалансированные классы (соотношения):
Взял за основу - соотношение Linux, как пример несбалансированного класса.

In [48]:
cat_cols = dataset.select_dtypes(include=['object', 'bool', 'category']).columns

for col in cat_cols:
    if dataset[col].nunique() <= 10:
        print(f"=== Распределение для: {col} ===")
        print((dataset[col].value_counts(normalize=True, dropna=False) * 100).round(2))
        print("-" * 30)

=== Распределение для: Windows ===
Windows
True     99.97
False     0.03
Name: proportion, dtype: float64
------------------------------
=== Распределение для: Mac ===
Mac
False    82.74
True     17.26
Name: proportion, dtype: float64
------------------------------
=== Распределение для: Linux ===
Linux
False    87.23
True     12.77
Name: proportion, dtype: float64
------------------------------


Выбросы и их процент:

In [12]:
def quantile(column_name):
  Q1 = dataset[column_name].quantile(0.25)
  Q3 = dataset[column_name].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  outsiders = dataset[(dataset[column_name] < lower_bound) | (dataset[column_name] > upper_bound)]
  return len(outsiders)


number_columns = dataset.select_dtypes(include=['float64', 'int64']).columns
for col in number_columns:
  total_count = len(dataset[col].dropna())
  if total_count == 0:
    continue  # если в колонке нет данных - пропускаю

  outsiders_count = quantile(col)
  percentage = round(outsiders_count / total_count * 100, 2)
  print(col, percentage)

Estimated owners 15.64
Peak CCU 1.02
Required age 7.35
Price 0.19
DiscountDLC count 14.33
Metacritic score 3.38
User score 0.03
Positive 16.91
Negative 16.45
Score rank 0.0
Achievements 6.84
Recommendations 16.7
Average playtime forever 20.8
Average playtime two weeks 3.14
Median playtime forever 20.8
Median playtime two weeks 3.14


Максимальный выброс - это 20.8 , можно считать, что 21%, столько и беру.

Количество числовых признаков:

In [28]:
len(dataset.select_dtypes(include=['float64', 'int64']).columns)

17

Хотя на сайте их 14.
Попробую вывести всё, что считает питон числовым признаком.

In [27]:
dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()

['Estimated owners',
 'Peak CCU',
 'Required age',
 'Price',
 'DLC count',
 'Metacritic score',
 'User score',
 'Positive',
 'Negative',
 'Score rank',
 'Achievements',
 'Recommendations',
 'Average playtime forever',
 'Average playtime two weeks',
 'Median playtime forever',
 'Median playtime two weeks',
 'Movies']

И здесь можно заметить различие:
- Estimated owners - на самом деле считается текстовым признаком
- Score rank - тоже string
- Movies - это массив (не число)
Поэтому числовых 14

Текстовых на сайте 14 - беру 14.

Все признаки ниже:


In [34]:
my_cols = [
    'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DLC count',
    'Metacritic score', 'User score', 'Positive', 'Negative', 'Score rank',
    'Achievements', 'Recommendations', 'Average playtime forever',
    'Average playtime two weeks', 'Median playtime forever',
    'Median playtime two weeks', 'Movies'
]



Категориальные признаки:

In [36]:
cat_cols = dataset.select_dtypes(include=['category', 'object', 'bool']).columns
print("Количество категориальных признаков:", len(cat_cols))
print("Список категориальных колонок:", cat_cols.tolist())

Количество категориальных признаков: 22
Список категориальных колонок: ['AppID', 'Name', 'Release date', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic url', 'Notes', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots']


Остальные признаки: их 6 на сайте:
AppID - идентификатор (ключ) (ID игры по сути)
Release date - дата (когда игра вышла)
Mac, Linux, Metacritic score - логические признаки
Achievements - достижения внутриигровые

Остальные признаки: ['Linux', 'Mac', 'Windows']
